In [1]:
import import_ipynb
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('punkt_tab')
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet as wn
from nltk.translate.bleu_score import sentence_bleu
import spacy
import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\krist\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
#Import functions from other notebook
from nlp_processing import text_preprocessing, nlp_processing, load_ingredients_list, tag_ingredients, identify_ingredients, perform_tfidf, perform_cosine_similarity

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\krist\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Total number of available ingredients:  1951
Detected Ingredient: mixed nut
567               Super Steak and Gravy
521    Foil Packet Veggies on the Grill
356                      Corn Chowder V
800        Quick French Onion Meatballs
452                    Orzo Pasta Salad
Name: name, dtype: object
BLUE Score: 1.5319719891192393e-231


In [13]:
def check_valid_ingredients(ingredient, ingredients_list):
    #Check if any valid ingredients were added
    if ingredient is None:
        print("Ingredient was not recognized. Please try again")
    #Check if ingredient is already on the list
    elif ingredient in ingredients_list:
         print("You have already added:", ingredient)
    else:
        #If valid add the ingredient to the user's ingredient list
        ingredients_list.append(ingredient)
        print("Current Ingredients List:", ingredients_list)

In [14]:
#Lemmatized verbs or commands e.g.:finished->finish
def lemmatized_verbs(token):
    wn = nltk.WordNetLemmatizer()
    text_lemmatized = []
    for word,tag in pos_tag(token):
        #Check if the token/word has a POS verb tag
        if tag[0] == "V" and word != "done":
             text_lemmatized.append(wn.lemmatize(word, tag[0].lower())) #Use the "v" tag to indicate that its a verb
        else:
            text_lemmatized.append(wn.lemmatize(word))
    return text_lemmatized

In [15]:
#Check if a token is synonymous/matches a specific command such as "finished" or "done"
def check_valid_commands(tokens, status):
    # Using Word2 net to find synonyms of the word "finish"
    command = "finish"
    synonyms = []
    for synonym in wn.synsets(command):
        for lemma in synonym.lemma_names():
            synonyms.append(lemma)
    synonyms.append("done") #Also add done to the synonym list
    
    #Ensure that the commands are lemmatized
    tokens = lemmatized_verbs(tokens)
    #Check if the user's token matches that of any word in the synonym list
    for token in tokens:
        if token in synonyms:
            status = "done"
            return status

In [16]:
def convert_words2number(token):
    words_to_nums = {
        "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
        "five": "5", "first": "1", "second": "2", "third": "3", "fourth": "4", "fifth": "5"
    }
    return words_to_nums.get(token)

In [28]:
#using Bleu score to match the recipe name given by the user to one of the top 5 recipe list
def get_bleu_score(self, tokens):
    tokens = [tokens]
    #Get the index positions of the top 5 recipes
    index = self.top5_recipe.index
    chosen_index = 0
    #Calculate the bleu score for each recipe list
    bleu_score = 0
    for i, recipe in enumerate(self.top5_recipe):
        #Perform text cleaning on recipe name
        recipe = text_preprocessing(recipe)
        recipe = word_tokenize(recipe)
        #Calculate the bleu score
        score = sentence_bleu(tokens, recipe)
        if score > bleu_score:
            bleu_score = score
            chosen_index = index[i]
            
    #If no matches found return
    if bleu_score == 0:
        return
    #Otherwise return the recipe's index with the highest bleu score
    else:
        return chosen_index

In [29]:
#
def check_recipe_response(tokens, self):
    #First, check if user used numbers/digit to pick a recipe from the list
    for token in tokens:
        #Check for worded numbers and convert them to digits
        number_choice = convert_words2number(token)
        if number_choice is not None:
            token = number_choice
        #Check for numbers/digits and check if this number is within range of 1-5
        if token.isnumeric() and int(token) in range(1,6):
            index = self.top5_recipe.index
            token = int(token) #Convert to int
            chosen_index = index[token - 1]
            #Return the dataframe index of the user's choice
            return chosen_index
    #Second, if user entered a name instead find a matchin one based on Bleu score   
    chosen_index = get_bleu_score(self, tokens)
    return chosen_index

In [167]:
def print_info(info, pattern):
    #Remove punctuations except for comma
    punctuations_with_comma = r'[\[\]]'
    info_list = re.sub(punctuations_with_comma, "", info) #Clean unwanted punctuations
    info_list = re.sub(r' +', " ", info_list) #Remove excessive whitespace
    # pattern = r'",|,"'
    info_list = re.sub(r"['|\"]", "", info_list) #Cleanup any remaining quotation marks
    info_list = re.split(pattern, info_list)
    #info_list = info_list.split(pattern)
    #Print the ingredient list
    i = 0
    for info in info_list:
        #Check if info is empty if so skip
        if len(info) == 0:
            continue
        print(f"{i+1}.) {info}")
        i += 1

In [168]:
#Dialogue Policy
class DialoguePolicy:
    def __init__(self):
        self.ingredients_list = []
        self.top5_recipe = []
        self.chosen_recipe_index = 0
        self.current_stage = "ingredients_collection"
        #Import the food dataset
        self.df_food = pd.read_csv("./dataset/Recipe_Dataset.csv")

    ####################3 main stages: ingredients collection, recipe choosing, recipe output
    ###Step 1.) Ingredients Collection Stage
    def choose_ingredient_stage(self):
        status = "ongoing"
        while status != "done":
            #Get user response
            response = input("Choose your ingredients:")
            #Apply text cleaning to response
            response = text_preprocessing(response)
            response,tokens = nlp_processing(response)
            #Check if the user's response contains a command
            status = check_valid_commands(tokens, status)
            if status == "done":
                break
            #Detect ingredients from text
            all_ingredients = load_ingredients_list()
            matcher, nlp = tag_ingredients(all_ingredients)
            ingredient = identify_ingredients(response, matcher, nlp)
            #Check if the detected ingredient is valid
            check_valid_ingredients(ingredient, self.ingredients_list)
    
        #Move on to the choosing recipe stage after finishing adding ingredients
        self.ingredients_list = [", ".join(self.ingredients_list)]
        self.current_stage = "choose_recipe_stage"
        print(self.ingredients_list)
        
    ###Step 2.) Choosing Recipe Stage
    def choose_recipe_stage(self):
        #Process the ingredients list and the dataset - tfidf
        df_tfidf, df_vector = perform_tfidf(self.df_food["ingredients"])
        #Apply Cosine similarity to get top 5 matching recipes
        self.top5_recipe = perform_cosine_similarity(df_tfidf, df_vector, self.ingredients_list)
        self.top5_recipe = self.df_food["name"][self.top5_recipe]
        for i,recipe in enumerate(self.top5_recipe):
            print(f"{i + 1}.) {recipe}")
        
        #Ask user to pick their preferred recipe from list
        status = "ongoing"
        while status != "done":
            #Get user response
            response = input("Which recipe would you like to see?:")
            #Clean the response
            response_cleaned = text_preprocessing(response)
            response_cleaned = word_tokenize(response_cleaned)
            self.chosen_recipe_index = check_recipe_response(response_cleaned, self)
            #Check if the response is valid
            if self.chosen_recipe_index is not None:
                status = "done"
                #Move on to step 3 Print Chosen Recipe
                self.current_stage = "show_recipe_stage"
                return
            else:
                print("I have not detected a matching recipe. Please try again")

    ###Step 3.) Print chosen recipe
    def print_recipe_stage(self):
        #Print the recipe name
        print(f"Here is the recipe for {self.df_food["name"].iloc[self.chosen_recipe_index]}:")
        #Print the ingredients list
        print("Ingredients: ")
        split_pattern = '","'
        print_info(self.df_food["ingredients_raw_str"].iloc[self.chosen_recipe_index], split_pattern)
        #Print the Step Instructions
        print("Instructions: ")
        split_pattern = "', '"
        print_info(self.df_food["steps"].iloc[self.chosen_recipe_index], split_pattern)
        status = "ongoing"
        while status != "done":
            response = input("Would you like to edit your 'ingredients' list or pick another 'recipe'")

In [171]:
####main
dialogue = DialoguePolicy()
while dialogue.current_stage != "quit":
    if dialogue.current_stage == "ingredients_collection":
        dialogue.choose_ingredient_stage()
    elif dialogue.current_stage == "choose_recipe_stage":
        dialogue.choose_recipe_stage()
    elif dialogue.current_stage == "show_recipe_stage":
        dialogue.print_recipe_stage()

Choose your ingredients: steak


Current Ingredients List: ['steak']


Choose your ingredients: eggs


Current Ingredients List: ['steak', 'egg']


Choose your ingredients: potatoes


...

Choose your ingredients: I'm finish adding ingredients


['steak, egg, potato, cabbage']
1.) Grandma Joyce's Split Pea Vegetable Soup
2.) Hearty Vegetable Soup
3.) Corn Chowder V
4.) Country Steak 'n' Gravy
5.) Pheasant Mulligan With Dumplings


Which recipe would you like to see?: I want to see recipe 1


Here is the recipe for Grandma Joyce's Split Pea Vegetable Soup:
Ingredients: 
1.) 1 large potato, chopped ,2 carrots, chopped ,2 stalks celery, chopped ,1/2 cup dried split peas, ham hock (for vegetarians ( 2 Tbs. butter),1 tablespoon salt, pepper,1/4 head cabbage,2 quarts water
Instructions: 
1.) Cook ham in water until really tender (you will want to cut the meat into very small pieces after cooking)., Add all other ingredients *except for cabbage., Cover and cook 30 minutes., Cut up cabbage and add to soup. Cook 15 more minutes.


KeyboardInterrupt: Interrupted by user